# FlowGuard User-Level Defense Experiments

This notebook evaluates **FlowGuardUserLevelDefense (C3)** against the new attack suite. The goal is to inspect where a per-user sliding-window distribution test helps beyond per-query FlowPure scoring.

## What This Defense Tests

C3 still computes a FlowPure-style per-query score, but its decision is user-level: once enough queries have accumulated, it compares the user's score distribution against benign reference scores with a KS statistic.

Expected behavior:
- It should help when attacks are individually subtle but collectively shifted.
- It should be relevant for sparse-probe behavior, where only a small fraction of a user's queries are malicious.
- It can miss early queries before the sliding window reaches `min_window`.
- It can behave similarly to FlowPure if the suite records mostly per-query scores rather than long multi-user sessions.

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve()
while not (project_root / "pyproject.toml").exists() and project_root != project_root.parent:
    project_root = project_root.parent
sys.path.insert(0, str(project_root / "notebooks" / "flow_matching_defenses"))

from flowguard_notebook_utils import (
    ATTACKS,
    apply_nature_style,
    build_single_defense_command,
    defense_rows,
    display_defense_intro,
    display_metric_table,
    extract_scores_from_experiment,
    experiment_result_path,
    plot_defense_metrics,
    plot_score_distributions,
    plot_utility_vs_detection,
    run_command,
    show_attack_query_grid,
)

apply_nature_style()
DEFENSE = "flowguard_userlevel"
RUN_ROOT = project_root / "runs" / "notebook" / "flowguard_suite"
RUN_EVALUATION = False
FORCE_RERUN = False
print(f"Project root: {project_root}")
print(f"Suite root: {RUN_ROOT}")

In [ ]:
display_defense_intro(DEFENSE)

In [ ]:
command = build_single_defense_command(
    project_root,
    RUN_ROOT,
    DEFENSE,
    attack_query_budget=248,
    benign_query_budget=248,
    force=FORCE_RERUN,
)
if RUN_EVALUATION:
    run_command(command, project_root)
else:
    print("RUN_EVALUATION=False, using existing artifacts if present.")
    print(" ".join(command))

In [ ]:
rows = defense_rows(RUN_ROOT, DEFENSE)
display_metric_table(rows)
plot_defense_metrics(rows, DEFENSE)
plot_utility_vs_detection(rows, DEFENSE)

## Score Distributions

These plots show the raw per-query score distributions that feed the user-level window. If the attack and benign distributions overlap heavily, C3 needs longer windows or stronger behavioral side channels to become reliable.

In [ ]:
plot_score_distributions(RUN_ROOT, DEFENSE)

## Window Flag Dynamics

C3 broadcasts the user-level decision back onto all queries in the current batch. The cumulative curve below shows when each attack starts being flagged during the recorded stream.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(8, 4))
for attack in ATTACKS:
    path = experiment_result_path(RUN_ROOT, DEFENSE, attack, benign=False)
    if path is None:
        continue
    _scores, flags, _threshold = extract_scores_from_experiment(path)
    if flags.size == 0:
        continue
    cumulative = np.cumsum(flags) / np.arange(1, flags.size + 1)
    ax.plot(cumulative, label=attack)
ax.set_title("C3 cumulative flagged fraction over attack stream")
ax.set_xlabel("query index")
ax.set_ylabel("cumulative flagged fraction")
ax.set_ylim(-0.02, 1.05)
ax.legend()
fig.tight_layout()
plt.show()

## Attack Query Visualization

The visual grid helps distinguish clean-query attacks from generator-style attacks. C3 itself is not an image-quality detector; it uses how a user's score distribution evolves over many submitted queries.

In [ ]:
for attack in ATTACKS:
    show_attack_query_grid(RUN_ROOT, DEFENSE, attack, count=6)

## Reading Your Current Results

In the current run, C3 behaves close to FlowPure for `a1_clean_transfer`, catches `a3_flowblind` and `maze_baseline`, but only partially catches `a2_projected_maze` and `adaptive_adaptive`. That is consistent with a windowed detector that still depends on the same underlying score: if denoising or adaptation keeps many scores near the benign region, the KS/window signal needs more queries or a stronger side channel to separate the user.